# Audio Classification with PyTorch and Kubeflow

This notebook demonstrates how to train a convolutional neural network (CNN) for audio classification using PyTorch. It is designed to be compatible with Kubeflow Trainer and can run both locally and in a distributed environment.

## Features
- **Kubeflow Compatible**: Uses environment variables for data and output paths.
- **Dynamic Class Handling**: Automatically detects classes from the directory structure.
- **Mel Spectrogram Transformation**: Converts audio waveforms into frequency-domain representations.
- **Non-Interactive**: Safe for automated execution in pipeline components.

## 1. Imports and Dependencies

We begin by importing the necessary libraries for audio processing, neural network construction, and data management.

In [83]:
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import soundfile as sf

from torch.utils.data import Dataset, DataLoader
from torchaudio.transforms import MelSpectrogram

## 2. Configuration

Define hyperparameters and set up paths. We use `os.environ.get` to allow these values to be overridden by Kubeflow environment variables.

In [84]:
DATA_DIR = os.environ.get("DATA_DIR", "./genres_original")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "./output")

BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-3
SAMPLE_RATE = 22050
AUDIO_LENGTH = 3  # seconds
FIXED_LENGTH = SAMPLE_RATE * AUDIO_LENGTH  # 66150 samples

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"DATA_DIR: {DEVICE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

Device: cpu
DATA_DIR: ./genres_original
OUTPUT_DIR: ./output


## 3. Dataset Implementation

The `AudioDataset` class handles loading audio files, resampling them to a consistent rate, and converting them into Mel Spectrograms.

In [85]:
class AudioDataset(Dataset):
    def __init__(self, root_dir):
        self.root = Path(root_dir)

        if not self.root.exists() or not self.root.is_dir():
            raise RuntimeError(f"DATA_DIR does not exist or is not a directory: {root_dir}")

        self.classes = sorted(
            p.name for p in self.root.iterdir()
            if p.is_dir() and not p.name.startswith(".")
        )

        if len(self.classes) < 2:
            raise RuntimeError(
                "DATA_DIR must contain at least two class subdirectories"
            )

        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        self.files = []
        for cls in self.classes:
            wavs = list((self.root / cls).glob("*.wav"))
            if not wavs:
                raise RuntimeError(f"No .wav files found in class folder: {cls}")
            self.files.extend(wavs)

        self.mel = MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_mels=64
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        wav_path = self.files[idx]
        label = self.class_to_idx[wav_path.parent.name]

        try:
            # Load audio using soundfile
            waveform, sr = sf.read(str(wav_path))
            waveform = torch.FloatTensor(waveform)
            
            # Handle stereo to mono conversion
            if waveform.ndim == 2:
                waveform = waveform.mean(dim=1)
            
            # Ensure it's 1D then add channel dimension
            if waveform.ndim == 1:
                waveform = waveform.unsqueeze(0)
            
            # Resample if needed
            if sr != SAMPLE_RATE:
                import torchaudio.functional as F
                waveform = F.resample(waveform, sr, SAMPLE_RATE)
            
            # Fix length: pad or truncate to FIXED_LENGTH
            if waveform.shape[1] > FIXED_LENGTH:
                waveform = waveform[:, :FIXED_LENGTH]
            elif waveform.shape[1] < FIXED_LENGTH:
                padding = FIXED_LENGTH - waveform.shape[1]
                waveform = torch.nn.functional.pad(waveform, (0, padding))

            mel = self.mel(waveform).squeeze(0)
            return mel, label
            
        except Exception as e:
            print(f"Error loading {wav_path}: {e}")
            # Return a random other sample instead
            return self.__getitem__((idx + 1) % len(self.files))

## 4. Model Architecture

We implement a Convolutional Neural Network (CNN) specifically tailored for spectrogram input. It uses multiple convolutional layers followed by adaptive pooling and a linear classifier.

In [86]:
class AudioCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # Convolutional feature extractor
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x):
        # Add channel dimension
        x = x.unsqueeze(1)
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

## 5. Training Function

The `train` function encapsulates the entire training process: data loading, model initialization, optimization, and saving the final model weights.

In [87]:
def train():
    # Prepare dataset and data loader
    dataset = AudioDataset(DATA_DIR)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    # Initialize model, optimizer, and loss criterion
    model = AudioCNN(num_classes=len(dataset.classes)).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    # Iterate through training epochs
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            # Optimization step
            optimizer.zero_grad()
            preds = model(x)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # Print average loss for this epoch
        print(f"Epoch {epoch + 1}/{EPOCHS} | Loss: {total_loss / len(loader):.4f}")

    # Save the trained model weights
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "model.pt"))
    print(f"Model saved to {OUTPUT_DIR}/model.pt")

## 6. Run Training

Execute the training process.

In [88]:
train()

Error loading genres_original\jazz\jazz.00054.wav: Error opening 'genres_original\\jazz\\jazz.00054.wav': Format not recognised.
Epoch 1/5 | Loss: 2.6958
Error loading genres_original\jazz\jazz.00054.wav: Error opening 'genres_original\\jazz\\jazz.00054.wav': Format not recognised.
Epoch 2/5 | Loss: 2.1627
Error loading genres_original\jazz\jazz.00054.wav: Error opening 'genres_original\\jazz\\jazz.00054.wav': Format not recognised.
Epoch 3/5 | Loss: 2.0905
Error loading genres_original\jazz\jazz.00054.wav: Error opening 'genres_original\\jazz\\jazz.00054.wav': Format not recognised.
Epoch 4/5 | Loss: 1.9702
Error loading genres_original\jazz\jazz.00054.wav: Error opening 'genres_original\\jazz\\jazz.00054.wav': Format not recognised.
Epoch 5/5 | Loss: 1.8855
Model saved to ./output/model.pt
